# Zeitliche Nachprüfung der kursweisen Hinweise

Wir prüfen die bereits festgelegte Modell- und Auswahlregel an einem späteren
Kursdurchlauf. Zunächst verwenden wir Kurseinträge aus 2013 als Training und
2014J als zeitlich spätere Prüfgruppe. Personen aus der Prüfgruppe werden aus
dem Training entfernt.

Auch diese Prüfung ist nicht vollständig unabhängig von bisherigen
Projektentscheidungen, da der OULAD-Datensatz bereits exploriert wurde.

In [1]:
from pathlib import Path
import pandas as pd

# Projektordner bestimmen, unabhängig davon, ob das Notebook
# vom Hauptordner oder aus "notebooks" gestartet wird.
projektordner = Path.cwd()
if projektordner.name.lower() == "notebooks":
    projektordner = projektordner.parent

datei = projektordner / "data" / "processed" / "oulad_tag28_modellbasis.csv"
basis = pd.read_csv(datei)

# Nur frühere Kursdurchläufe für das Training verwenden.
training = basis[
    basis["code_presentation"].isin(["2013B", "2013J"])
].copy()

pruefung = basis[
    basis["code_presentation"].eq("2014J")
].copy()

# Dieselbe Person soll nicht in beiden Gruppen vorkommen.
gemeinsame_personen = (
    set(training["id_student"]) & set(pruefung["id_student"])
)
training = training[
    ~training["id_student"].isin(pruefung["id_student"])
].copy()

assert not training.empty and not pruefung.empty
assert set(training["id_student"]).isdisjoint(pruefung["id_student"])

# Größe und Zielanteil für jedes Modul sichtbar machen.
ueberblick = pd.concat(
    [
        training.assign(gruppe="Training 2013"),
        pruefung.assign(gruppe="Prüfung 2014J"),
    ],
    ignore_index=True,
)

ueberblick = (
    ueberblick.groupby(["gruppe", "code_module"])
    .agg(
        kurseintraege=("id_student", "size"),
        abbruch_oder_fail=("abbruch_oder_fail", "sum"),
    )
    .reset_index()
)

ueberblick["zielanteil_prozent"] = (
    100
    * ueberblick["abbruch_oder_fail"]
    / ueberblick["kurseintraege"]
).round(1)

print("Wegen Überschneidung ausgeschlossene Personen:", len(gemeinsame_personen))
print("Training:", len(training), "| Prüfung:", len(pruefung))
print(
    "Module ohne frühere Trainingsdaten:",
    sorted(set(pruefung["code_module"]) - set(training["code_module"]))
)
display(ueberblick)

Wegen Überschneidung ausgeschlossene Personen: 756
Training: 11031 | Prüfung: 9123
Module ohne frühere Trainingsdaten: ['CCC']


,gruppe,code_module,kurseintraege,abbruch_oder_fail,zielanteil_prozent
0,Prüfung 2014J,AAA,348,95,27.3
1,Prüfung 2014J,BBB,1773,622,35.1
2,Prüfung 2014J,CCC,1990,976,49.0
3,Prüfung 2014J,DDD,1447,655,45.3
4,Prüfung 2014J,EEE,1016,332,32.7
5,Prüfung 2014J,FFF,1834,717,39.1
6,Prüfung 2014J,GGG,715,271,37.9
7,Training 2013,AAA,339,61,18.0
8,Training 2013,BBB,3316,1444,43.5
9,Training 2013,DDD,2496,1325,53.1


## Modelltest auf 2014J

Die Modelle lernen ausschließlich aus den ausgewählten Kursdurchläufen von
2013. Bewertet werden ihre Rangfolgen für 2014J. Hinweise werden getrennt
innerhalb jeder Kurspräsentation vergeben. Zusätzlich betrachten wir jedes
Modul einzeln und vergleichen mit der einfachen Regel „fälliges Assessment
bis Tag 28 nicht erfüllt“.

In [2]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

# Exakt die Merkmale aus dem bisherigen Tag-28-Modellvergleich.
kategorien = ["code_module", "code_presentation"]
grundmerkmale = kategorien + ["anmeldetag"]

vle_merkmale = [
    "klicks_tag_0_bis_6",
    "klicks_tag_7_bis_14",
    "klicks_tag_15_bis_28",
]

assessment_merkmale = [
    "assessments_faellig_bis28",
    "abgaben_bis28",
    "assessments_fehlend_bis28",
    "banked_faellige_bis28",
]

alle_merkmale = grundmerkmale + vle_merkmale + assessment_merkmale

def baue_modell(merkmale, klassifikator):
    """Erstellt die gleiche Aufbereitung wie im bisherigen Modellvergleich."""
    zahlen = [spalte for spalte in merkmale if spalte not in kategorien]

    vorbereitung = ColumnTransformer([
        (
            "kategorien",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            kategorien,
        ),
        ("zahlen", StandardScaler(), zahlen),
    ])

    return Pipeline([
        ("vorbereitung", vorbereitung),
        ("klassifikator", klassifikator),
    ])

ziel_training = training["abbruch_oder_fail"].astype(int)
ziel_pruefung = pruefung["abbruch_oder_fail"].astype(int).to_numpy()

modelle = {
    "Kurs + Anmeldung": (
        grundmerkmale,
        LogisticRegression(max_iter=1000),
    ),
    "Boosting: Alle Merkmale": (
        alle_merkmale,
        HistGradientBoostingClassifier(random_state=42),
    ),
}

risikowerte = {}

for name, (merkmale, klassifikator) in modelle.items():
    modell = baue_modell(merkmale, klassifikator)

    # Lernen nur mit 2013; Vorhersage für 2014J.
    modell.fit(training[merkmale], ziel_training)
    risikowerte[name] = modell.predict_proba(
        pruefung[merkmale]
    )[:, 1]

    print(
        name,
        "| AP:",
        round(average_precision_score(
            ziel_pruefung, risikowerte[name]
        ), 3),
    )

print("Zielanteil 2014J:", round(ziel_pruefung.mean(), 3))

Kurs + Anmeldung | AP: 0.424
Boosting: Alle Merkmale | AP: 0.635
Zielanteil 2014J: 0.402


In [3]:
ergebnisse = []
diagnose_10 = None

for name, risiko in risikowerte.items():
    assert len(risiko) == len(pruefung)

    for anteil in (10, 20):
        hinweis = np.zeros(len(pruefung), dtype=bool)

        # Die höchsten Werte innerhalb jeder Kurspräsentation auswählen.
        for _, positionen in pruefung.groupby(
            ["code_module", "code_presentation"]
        ).indices.items():
            positionen = np.asarray(positionen)
            anzahl = int(np.ceil(len(positionen) * anteil / 100))

            sortiert = positionen[
                np.argsort(-risiko[positionen], kind="stable")
            ]
            hinweis[sortiert[:anzahl]] = True

        anzahl_hinweise = int(hinweis.sum())
        treffer = int(ziel_pruefung[hinweis].sum())

        ergebnisse.append({
            "Verfahren": name,
            "Quote je Kurs": f"{anteil} %",
            "Hinweise": anzahl_hinweise,
            "Treffer": treffer,
            "Trefferquote %": round(
                100 * treffer / anzahl_hinweise, 1
            ),
            "Recall %": round(
                100 * treffer / ziel_pruefung.sum(), 1
            ),
        })

        # Für das ausgewählte Modell die 10-%-Hinweise je Modul merken.
        if name == "Boosting: Alle Merkmale" and anteil == 10:
            diagnose_10 = pruefung[
                ["code_module", "abbruch_oder_fail"]
            ].reset_index(drop=True).copy()
            diagnose_10["hinweis"] = hinweis
            diagnose_10["treffer"] = hinweis & (ziel_pruefung == 1)

display(pd.DataFrame(ergebnisse))

# Prüfen, ob einzelne Module trotz guter Gesamtwerte schwach abschneiden.
je_modul = diagnose_10.groupby("code_module").agg(
    kurseintraege=("hinweis", "size"),
    spaetere_faelle=("abbruch_oder_fail", "sum"),
    hinweise=("hinweis", "sum"),
    treffer=("treffer", "sum"),
)

je_modul["trefferquote_%"] = (
    100 * je_modul["treffer"] / je_modul["hinweise"]
).round(1)
je_modul["recall_%"] = (
    100 * je_modul["treffer"] / je_modul["spaetere_faelle"]
).round(1)

display(je_modul)

# Die einfache Assessment-Regel hat keine feste 10-%-Quote.
regel = pruefung["assessments_fehlend_bis28"].gt(0).to_numpy()
regel_treffer = int(ziel_pruefung[regel].sum())

print(
    "Assessment-Regel | Hinweise:", int(regel.sum()),
    "| Treffer:", regel_treffer,
    "| Trefferquote %:", round(100 * regel_treffer / regel.sum(), 1),
    "| Recall %:", round(
        100 * regel_treffer / ziel_pruefung.sum(), 1
    ),
)

,Verfahren,Quote je Kurs,Hinweise,Treffer,Trefferquote %,Recall %
0,Kurs + Anmeldung,10 %,915,400,43.7,10.9
1,Kurs + Anmeldung,20 %,1827,802,43.9,21.9
2,Boosting: Alle Merkmale,10 %,915,729,79.7,19.9
3,Boosting: Alle Merkmale,20 %,1827,1225,67.0,33.4


,kurseintraege,spaetere_faelle,hinweise,treffer,trefferquote_%,recall_%
code_module,,,,,,
AAA,348,95,35,26,74.3,27.4
BBB,1773,622,178,120,67.4,19.3
CCC,1990,976,199,165,82.9,16.9
DDD,1447,655,145,127,87.6,19.4
EEE,1016,332,102,84,82.4,25.3
FFF,1834,717,184,157,85.3,21.9
GGG,715,271,72,50,69.4,18.5


Assessment-Regel | Hinweise: 634 | Treffer: 528 | Trefferquote %: 83.3 | Recall %: 14.4


### Zwischenfazit

Im zeitlich späteren Kursdurchlauf 2014J erreicht das Boosting bei
kursweise 10 % Hinweisen eine Trefferquote von 79,7 % und einen Recall
von 19,9 %. Es erkennt 729 spätere Fälle bei 915 Hinweisen. Das
Referenzmodell aus Kurs- und Anmeldedaten erkennt bei gleicher
Hinweiszahl 400 Fälle.

Die einfache Assessment-Regel erzeugt 634 Hinweise mit 528 Treffern.
Ihre Kennzahlen sind wegen der anderen Hinweiszahl nicht unmittelbar
mit der 10-%-Auswahl vergleichbar.

CCC besitzt keine Trainingsdaten aus 2013. Das Ergebnis dieses Moduls
wird deshalb gesondert betrachtet. Die zeitliche Nachprüfung stärkt die
Einschätzung für diesen historischen Datensatz, ersetzt aber keine
Prüfung mit Daten einer tatsächlichen Weiterbildungseinrichtung.

In [4]:
# Späteres Kursergebnis ausschließlich für die Diagnose einlesen.
info = pd.read_csv(
    projektordner / "Daten OULAD" / "studentInfo.csv",
    usecols=[
        "code_module", "code_presentation",
        "id_student", "final_result",
    ],
)

schluessel = ["code_module", "code_presentation", "id_student"]

diagnose = pruefung[
    schluessel + ["assessments_fehlend_bis28"]
].reset_index(drop=True).merge(
    info,
    on=schluessel,
    how="left",
    validate="one_to_one",
    sort=False,
)

# Prüfen, dass die Ergebniswerte zu den Modellvorhersagen passen.
assert diagnose[schluessel].equals(
    pruefung[schluessel].reset_index(drop=True)
)
assert np.array_equal(
    diagnose["final_result"].isin(["Fail", "Withdrawn"]).to_numpy(),
    ziel_pruefung == 1,
)

diagnose["modell_hinweis"] = diagnose_10["hinweis"].to_numpy()
diagnose["regel_hinweis"] = (
    diagnose["assessments_fehlend_bis28"] > 0
)

# Wie viele Pass-, Fail- und Withdrawn-Fälle markiert das Modell?
nach_ergebnis = diagnose.groupby("final_result").agg(
    kurseintraege=("id_student", "size"),
    mit_modellhinweis=("modell_hinweis", "sum"),
)
nach_ergebnis["anteil_mit_hinweis_%"] = (
    100
    * nach_ergebnis["mit_modellhinweis"]
    / nach_ergebnis["kurseintraege"]
).round(1)

display(nach_ergebnis)

# Prüfen, welche Fälle nur das Modell beziehungsweise nur die Regel findet.
modell = diagnose["modell_hinweis"].to_numpy()
regel = diagnose["regel_hinweis"].to_numpy()

diagnose["vergleich"] = np.select(
    [modell & regel, modell & ~regel, ~modell & regel],
    ["Beide", "Nur Modell", "Nur Regel"],
    default="Keiner",
)

ueberschneidung = diagnose.groupby("vergleich").agg(
    kurseintraege=("id_student", "size"),
    spaetere_faelle=("final_result", lambda x:
                    x.isin(["Fail", "Withdrawn"]).sum()),
)

ueberschneidung["trefferquote_%"] = (
    100
    * ueberschneidung["spaetere_faelle"]
    / ueberschneidung["kurseintraege"]
).round(1)

display(
    ueberschneidung.reindex(
        ["Beide", "Nur Modell", "Nur Regel", "Keiner"]
    )
)

,kurseintraege,mit_modellhinweis,anteil_mit_hinweis_%
final_result,,,
Distinction,1164,20,1.7
Fail,1975,421,21.3
Pass,4291,166,3.9
Withdrawn,1693,308,18.2


,kurseintraege,spaetere_faelle,trefferquote_%
vergleich,,,
Beide,590,501,84.9
Nur Modell,325,228,70.2
Nur Regel,44,27,61.4
Keiner,8164,2912,35.7


### Zusatznutzen gegenüber der einfachen Assessment-Regel

Bei kursweise 10 % Modellhinweisen markieren Modell und einfache
Assessment-Regel gemeinsam 590 Kurseinträge; 501 davon enden später
mit `Fail` oder `Withdrawn`.

Das Modell markiert darüber hinaus 325 Kurseinträge, welche die Regel
nicht erfasst. Darunter sind 228 spätere Fälle (70,2 % Trefferquote).
Nur die Regel markiert weitere 44 Einträge, darunter 27 spätere Fälle.

Das Modell erkennt 421 von 1.975 späteren `Fail`-Fällen (21,3 %) und
308 von 1.693 späteren Abmeldungen (18,2 %). Es markiert auch 186
Einträge mit späterem `Pass` oder `Distinction`. Ein Hinweis ist daher
ein Anlass für eine unterstützende Prüfung, keine Feststellung über
den weiteren Verlauf.